<a href="https://colab.research.google.com/github/shivanshi-09/IML_Midterm/blob/main/MDP_Diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product

In [ ]:
classes = ['Healthy', 'Prediabetic', 'Diabetic']
tiers = [
    'T1: Basic screening',
    'T1+T2: Blood panel',
    'T1+T2+T3: Full biochemistry'
]

tier_costs = np.array([0, 1000, 2500])
clinical_cost = np.array([
    [0, 1, 2],
    [1, 0, 1],
    [5, 2, 0]
], dtype=float)

clinical_scale = 10000.0

In [ ]:
CM = {
    'T1: Basic screening': np.array([
        [0.684, 0.187, 0.128],
        [0.251, 0.355, 0.394],
        [0.153, 0.387, 0.460]
    ]),
    'T1+T2: Blood panel': np.array([
        [0.706, 0.220, 0.074],
        [0.235, 0.508, 0.257],
        [0.093, 0.427, 0.480]
    ]),
    'T1+T2+T3: Full biochemistry': np.array([
        [0.724, 0.227, 0.049],
        [0.278, 0.480, 0.242],
        [0.127, 0.340, 0.533]
    ]),
}
class_prior = np.array([0.58, 0.27, 0.15])

In [ ]:
def emc(cm_prob, class_prior, clinical_cost, scale):
    total = 0.0
    for true_cls in range(3):
        for pred_cls in range(3):
            total += (class_prior[true_cls]
                      * cm_prob[true_cls, pred_cls]
                      * clinical_cost[true_cls, pred_cls]
                      * scale)
    return total
print("===================================")
print("Expected cost per patient per tier")
print("===================================")
expected_costs = {}
for tier in tiers:
    ec = emc(CM[tier], class_prior, clinical_cost, clinical_scale)
    expected_costs[tier] = ec
    print(f"  {tier:<15}  ₹{ec:>7.1f}")
print()


Expected cost per patient per tier
  T1: Basic screening  ₹ 6619.4
  T1+T2: Blood panel  ₹ 5441.3
  T1+T2+T3: Full biochemistry  ₹ 5261.5



In [ ]:
gamma = 1.0
incremental_cost = np.array([0,1000,1500])
def total_cost(state, action):
  tier_name = tiers[state]
  if action == 'stop':
    return expected_costs[tier_name]
  elif action == 'acquire':
    if state == 2:
      return expected_costs[tier_name]
    next_state = state + 1
    next_tier = tiers[next_state]
    acq_cost = incremental_cost[next_state]
    misclass_cost = expected_costs[next_tier]
    return acq_cost + misclass_cost

In [ ]:
def value_iteration(gamma=1.0, theta=1e-6, max_iter=1000):
    V      = np.zeros(3)
    policy = [''] * 3
    for iteration in range(max_iter):
        V_new = np.zeros(3)
        for s in range(3):
            costs = {'stop': total_cost(s, 'stop')}
            if s < 2:
                costs['acquire'] = incremental_cost[s + 1] + gamma * V[s + 1]
            best_action = min(costs, key=costs.get)
            policy[s]   = best_action
            V_new[s]    = costs[best_action]
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    return V, policy, iteration + 1

V, policy, iters = value_iteration()

In [ ]:
print("===============================")
print("Finite-horizon diagnostic policy")
print(f"Converged in {iters} iterations")
print("===============================")

print(f"\n{'State':<35} {'Optimal Action':<20} {'Minimum Expected Cost (₹)'}")
print("-" * 80)

for s in range(3):
    print(f"  {tiers[s]:<33} {policy[s]:<20} ₹{V[s]:>8.1f}")

print()
print("=" * 60)
print("Marginal Value of Additional Diagnostic Testing")
print("=" * 60)

for s in range(2):

    tier_now  = tiers[s]
    tier_next = tiers[s + 1]

    cost_reduction = expected_costs[tier_now] - expected_costs[tier_next]

    acq_cost = incremental_cost[s + 1]

    net_benefit = cost_reduction - acq_cost

    print(f"\n  {tier_now} → {tier_next}")

    print(f"Acquisition cost: ₹{acq_cost:>7.1f}")

    print(f"Expected reduction in clinical error cost: ₹{cost_reduction:>7.1f}")

    print(f"Net economic benefit: ₹{net_benefit:>7.1f}  "
          f"({'ACQUIRE' if net_benefit > 0 else 'STOP — not cost-effective'})")

print()
print("Note:")
print("Because transitions are deterministic and the horizon is finite,")
print("the optimal MDP policy reduces to sequential marginal")
print("cost-benefit comparison between adjacent diagnostic tiers.")

Finite-horizon diagnostic policy
Converged in 4 iterations

State                               Optimal Action       Minimum Expected Cost (₹)
--------------------------------------------------------------------------------
  T1: Basic screening               acquire              ₹  6441.3
  T1+T2: Blood panel                stop                 ₹  5441.3
  T1+T2+T3: Full biochemistry       stop                 ₹  5261.5

Marginal Value of Additional Diagnostic Testing

  T1: Basic screening → T1+T2: Blood panel
Acquisition cost: ₹ 1000.0
Expected reduction in clinical error cost: ₹ 1178.1
Net economic benefit: ₹  178.1  (ACQUIRE)

  T1+T2: Blood panel → T1+T2+T3: Full biochemistry
Acquisition cost: ₹ 1500.0
Expected reduction in clinical error cost: ₹  179.8
Net economic benefit: ₹-1320.2  (STOP — not cost-effective)

Note:
Because transitions are deterministic and the horizon is finite,
the optimal MDP policy reduces to sequential marginal
cost-benefit comparison between adjacent dia

In [ ]:
print("=" * 75)
print("Sensitivity Analysis: Cost of Missing a Diabetic Patient")
print("=" * 75)

print(f"{'C[2,0]':<10} "
      f"{'Saving T1→T2':>16} "
      f"{'vs ₹1000':>12} "
      f"{'T1 policy':>14} "
      f"{'T1+T2 policy':>16}")

print("-" * 75)
flip_point = None
for diabetic_miss_cost in [5, 10, 20, 50, 100, 133, 200, 371]:
    cost_matrix = clinical_cost.copy()
    cost_matrix[2, 0] = diabetic_miss_cost
    ec_s = {}
    for tier in tiers:
        ec_s[tier] = emc(CM[tier], class_prior, cost_matrix, clinical_scale)

    saving_12 = (ec_s['T1: Basic screening'] - ec_s['T1+T2: Blood panel'])
    saving_23 = (ec_s['T1+T2: Blood panel'] - ec_s['T1+T2+T3: Full biochemistry'])

    pol0 = ('ACQUIRE' if saving_12 > incremental_cost[1] else 'STOP')
    pol1 = ('ACQUIRE' if saving_23 > incremental_cost[2] else 'STOP')
    if pol0 == 'ACQUIRE' and flip_point is None:
        flip_point = diabetic_miss_cost
    flag = " ← POLICY FLIP" if pol0 == 'ACQUIRE' else ""
    print(f"  {diabetic_miss_cost:<8} "
          f"{saving_12:>16.1f} "
          f"{incremental_cost[1]:>12} "
          f"{pol0:>14} "
          f"{pol1:>16}{flag}")
print()
if flip_point is not None:
    print(f"Approximate break-even point:")
    print(f"When C[2,0] ≳ {flip_point}, acquiring Tier 2 becomes economically justified.")
else:
    print("No policy flip observed within tested range.")

Sensitivity Analysis: Cost of Missing a Diabetic Patient
C[2,0]         Saving T1→T2     vs ₹1000      T1 policy     T1+T2 policy
---------------------------------------------------------------------------
  5                  1178.1         1000        ACQUIRE             STOP ← POLICY FLIP
  10                 1628.1         1000        ACQUIRE             STOP ← POLICY FLIP
  20                 2528.1         1000        ACQUIRE             STOP ← POLICY FLIP
  50                 5228.1         1000        ACQUIRE             STOP ← POLICY FLIP
  100                9728.1         1000        ACQUIRE             STOP ← POLICY FLIP
  133               12698.1         1000        ACQUIRE             STOP ← POLICY FLIP
  200               18728.1         1000        ACQUIRE             STOP ← POLICY FLIP
  371               34118.1         1000        ACQUIRE             STOP ← POLICY FLIP

Approximate break-even point:
When C[2,0] ≳ 5, acquiring Tier 2 becomes economically justified.


In [ ]:
print("=== Policy comparison ===")
policies = {
    'Never test (stop at T1)': ['stop', 'stop', 'stop'],
    'Always full panel (T3)': ['acquire', 'acquire', 'stop'],
    'Always one tier up': ['acquire', 'stop', 'stop'],
    'MDP optimal': policy,
}

print(f"{'Policy':<28} {'Expected total cost (₹)':>25}")
print("-" * 55)
for name, pol in policies.items():
    s, total = 0, 0.0
    while True:
        a = pol[s]
        if a == 'stop' or s == 2:
            total += expected_costs[tiers[s]]
            break
        else:
            total += incremental_cost[s + 1]
            s += 1
    print(f"{name:<26} ₹{total:>10.1f}")

=== Policy comparison ===
Policy                         Expected total cost (₹)
-------------------------------------------------------
Never test (stop at T1)    ₹    6619.4
Always full panel (T3)     ₹    7761.5
Always one tier up         ₹    6441.3
MDP optimal                ₹    6441.3
